# Discovery + Silver: `billing.subscriptions`

La tabla con la regla de calidad mas importante del dominio billing. Antes de decidir la regla, medimos exactamente que tan grave es el problema (mismo criterio que usamos con `courses.department`: no asumir, medir).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.billing__subscriptions", engine)
df.shape

(15000, 9)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

subscription_id            object
status                     object
start_date                 object
end_date                   object
customer_id                object
product_id                 object
_source_file               object
_ingested_at       datetime64[ns]
_dag_run_id                object
dtype: object


,subscription_id,status,start_date,end_date,customer_id,product_id,_source_file,_ingested_at,_dag_run_id
0,SUB-0000001,active,2024-10-15,2026-05-14,CUS-0006082,PRD-00154,billing/subscriptions.csv,2026-07-17 15:16:32.843550,manual__2026-07-17T15:16:30+00:00
1,SUB-0000002,cancelled,2023-12-22,2025-03-04,CUS-0001818,PRD-00158,billing/subscriptions.csv,2026-07-17 15:16:32.843550,manual__2026-07-17T15:16:30+00:00
2,SUB-0000003,active,2021-02-27,2025-01-25,CUS-0004408,PRD-00064,billing/subscriptions.csv,2026-07-17 15:16:32.843550,manual__2026-07-17T15:16:30+00:00
3,SUB-0000004,active,2020-10-02,2025-05-23,CUS-0005220,PRD-00112,billing/subscriptions.csv,2026-07-17 15:16:32.843550,manual__2026-07-17T15:16:30+00:00
4,SUB-0000005,paused,2021-06-11,2024-04-10,CUS-0008108,PRD-00123,billing/subscriptions.csv,2026-07-17 15:16:32.843550,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("subscription_id duplicados:", df["subscription_id"].duplicated().sum())

customers = pd.read_sql("SELECT customer_id FROM silver.billing__customers", engine)
products = pd.read_sql("SELECT product_id FROM silver.billing__products", engine)
print("customer_id huerfanos:", (~df["customer_id"].isin(customers["customer_id"])).sum())
print("product_id huerfanos:", (~df["product_id"].isin(products["product_id"])).sum())

Nulos por columna:
subscription_id    0
status             0
start_date         0
end_date           0
customer_id        0
product_id         0
_source_file       0
_ingested_at       0
_dag_run_id        0
dtype: int64

subscription_id duplicados: 0
customer_id huerfanos: 0
product_id huerfanos: 0


## 3. El problema de fechas: medirlo exactamente

Dos observaciones distintas de `docs/calidad_datos.md` que hay que medir por separado antes de decidir que hacer:

1. `start_date > end_date` (imposible logicamente: una suscripcion no puede terminar antes de empezar).
2. `end_date` poblado en suscripciones `active` (posible defecto, o posible que `end_date` sea simplemente "fecha de fin contratada" y no "fecha de terminacion real" -- hay que ver si pasa en el 100% de los casos o solo en algunos).

In [4]:
start = pd.to_datetime(df["start_date"])
end = pd.to_datetime(df["end_date"])

print("Valores de status:")
print(df["status"].value_counts())
print()

invertidas = start > end
print("start_date > end_date:", invertidas.sum(), f"({invertidas.mean()*100:.1f}%)")
print()

print("% de filas con end_date poblado, por status:")
print(df.assign(end_populated=end.notna()).groupby("status")["end_populated"].mean() * 100)

Valores de status:
status
active       11272
cancelled     2242
paused        1486
Name: count, dtype: int64

start_date > end_date: 783 (5.2%)

% de filas con end_date poblado, por status:
status
active       100.0
cancelled    100.0
paused       100.0
Name: end_populated, dtype: float64


## 4. Conclusion

`end_date` esta poblado en el 100% de las filas **para todos los status por igual** (no solo `active`) -- eso indica que es simplemente el diseño del campo ("fecha de fin contratada", no "fecha real de terminacion"), no un defecto especifico de las suscripciones activas. **No se anula por esa razon.**

El unico problema real y medible es `start_date > end_date` (imposible logicamente). Regla de limpieza:

- Cuando `start_date > end_date`: se marca la fila con `_end_date_invalidated = True` y se anula `end_date` (queda `NULL` en silver). El dato original sigue disponible en `bronze` para auditoria.
- `status` -> `strip()` + minusculas.
- Fechas -> castear a `date` real.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["subscription_id", "customer_id", "product_id", "status", "start_date", "end_date"]].copy()

df_silver["status"] = df_silver["status"].str.strip().str.lower()
df_silver["start_date"] = pd.to_datetime(df_silver["start_date"]).dt.date
df_silver["end_date"] = pd.to_datetime(df_silver["end_date"]).dt.date

invalid_end = df_silver["start_date"] > df_silver["end_date"]
df_silver["_end_date_invalidated"] = invalid_end
df_silver.loc[invalid_end, "end_date"] = None

print("Filas con end_date invalidado:", invalid_end.sum())
df_silver.head()

Filas con end_date invalidado: 783


,subscription_id,customer_id,product_id,status,start_date,end_date,_end_date_invalidated
0,SUB-0000001,CUS-0006082,PRD-00154,active,2024-10-15,2026-05-14,False
1,SUB-0000002,CUS-0001818,PRD-00158,cancelled,2023-12-22,2025-03-04,False
2,SUB-0000003,CUS-0004408,PRD-00064,active,2021-02-27,2025-01-25,False
3,SUB-0000004,CUS-0005220,PRD-00112,active,2020-10-02,2025-05-23,False
4,SUB-0000005,CUS-0008108,PRD-00123,paused,2021-06-11,2024-04-10,False


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert df_silver["subscription_id"].is_unique
assert df_silver["customer_id"].isin(customers["customer_id"]).all()
assert df_silver["product_id"].isin(products["product_id"]).all()
# ya no deberia quedar ningun start_date > end_date entre las filas con end_date no nulo
remaining = df_silver.dropna(subset=["end_date"])
assert (remaining["start_date"] <= remaining["end_date"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 15000 filas listas para silver


## 7. Escribir en `silver.billing__subscriptions`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "billing__subscriptions",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.billing__subscriptions")

Escrito en silver.billing__subscriptions


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.billing__subscriptions LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(*) FILTER (WHERE _end_date_invalidated) AS invalidadas, count(end_date) AS con_end_date FROM silver.billing__subscriptions", engine))
check

   filas  invalidadas  con_end_date
0  15000          783         14217


,subscription_id,customer_id,product_id,status,start_date,end_date,_end_date_invalidated,_silver_loaded_at
0,SUB-0000001,CUS-0006082,PRD-00154,active,2024-10-15,2026-05-14,False,2026-07-17 15:17:12.239875+00:00
1,SUB-0000002,CUS-0001818,PRD-00158,cancelled,2023-12-22,2025-03-04,False,2026-07-17 15:17:12.239875+00:00
2,SUB-0000003,CUS-0004408,PRD-00064,active,2021-02-27,2025-01-25,False,2026-07-17 15:17:12.239875+00:00
3,SUB-0000004,CUS-0005220,PRD-00112,active,2020-10-02,2025-05-23,False,2026-07-17 15:17:12.239875+00:00
4,SUB-0000005,CUS-0008108,PRD-00123,paused,2021-06-11,2024-04-10,False,2026-07-17 15:17:12.239875+00:00
